In [4]:
import os
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END

In [5]:
# Define shared transactional notebook data boundaries
class AgentTraceState(TypedDict):
    input_query: str
    classified_intent: str
    agent_response: str
    next_step: str

# Define strict routing structures
class IntentSchema(BaseModel):
    intent: Literal["billing", "technical"] = Field(description="The primary category of the inbound customer inquiry.")

# Initialize the OpenAI model core
llm = ChatOpenAI(model="gpt-4o", temperature=0.1)


In [6]:
# Node 1: Direct Classification Engine
def classification_node(state: AgentTraceState):
    print("[Node] Categorizing customer inquiry intent...")
    system_prompt = "You are an intake triage system. Classify the user query into billing or technical labels."
    
    # LangSmith automatically intercepts and visualizes this structured output call
    structured_llm = llm.with_structured_output(IntentSchema)
    result = structured_llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=state["input_query"])])
    
    # Update next_step to match the chosen target queue dynamically
    next_queue = "billing_queue" if result.intent == "billing" else "technical_queue"
    return {"classified_intent": result.intent, "next_step": next_queue}

# Node 2: Specialized Processing
def billing_handler_node(state: AgentTraceState):
    print("[Node] Executing account ledger analysis steps...")
    return {"agent_response": "Processed transaction audit records safely.", "next_step": "finish"}

# Router Edge function mapping graph trajectories
def route_by_intent(state: AgentTraceState) -> Literal["billing_queue", "finish"]:
    return state["next_step"]


In [7]:

# Construct and compile the network layout graph
builder = StateGraph(AgentTraceState)
builder.add_node("classifier", classification_node)
builder.add_node("billing_queue", billing_handler_node)

builder.add_edge(START, "classifier")

# 1. First conditional edge (This one is correct)
builder.add_conditional_edges(
    "classifier",
    route_by_intent,
    {
        "billing_queue": "billing_queue",
        "finish": END
    }
)

# 2. FIXED: Changed route_next_step to route_by_intent
builder.add_conditional_edges(
    "billing_queue", 
    route_by_intent, # <-- FIX IS HERE
    {
        "finish": END
    }
)

app = builder.compile()
# Triggering execution. LangSmith catches this entire trace tree automatically.
if __name__ == "__main__":
    print(" Running telemetry-tracked agent transaction...")
    payload = {
        "input_query": "Why was my corporate credit card billed double for subscription services yesterday?",
        "classified_intent": "",
        "agent_response": "",
        "next_step": ""
    }
    config = {"configurable": {"thread_id": "linkedin_demo_trace_1"}}
    app.invoke(payload, config)
    print(" Run complete. Log details are visible inside your LangSmith Web Console Dashboard.")

 Running telemetry-tracked agent transaction...
[Node] Categorizing customer inquiry intent...
[Node] Executing account ledger analysis steps...
 Run complete. Log details are visible inside your LangSmith Web Console Dashboard.
